In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from scipy import stats, fft


# Time-domain data
def timeFeatures(data):
    feature = []  # initialize feature list
    for i in range(len(data)):
        mean = np.mean(data[i])  # mean
        std = np.std(data[i])  # standard deviation
        rms = np.sqrt(np.mean(data[i] ** 2))  # root mean squre
        peak = np.max(abs(data[i]))  # peak
        skew = stats.skew(data[i])  # skewness
        kurt = stats.kurtosis(data[i])  # kurtosis
        cf = peak / rms  # crest factor
        # number of feature of each measurement = 7
        feature.append(
            np.array([mean, std, rms, peak, skew, kurt, cf], dtype=float)
        )
    feature = np.array(feature)
    return feature  # feature list, each element is numpy array with datatype float


# DFT magnitude data
def freqFeatures(data):
    feature = []
    for i in range(len(data)):
        N = len(data[i])  # number of data
        yf = (
            2 / N * np.abs(fft.fft(data[i])[: N // 2])
        )  # yf is DFT signal magnitude
        yf[0] = 0
        feature.append(np.array(yf))
    feature = np.array(feature)
    return feature


def tensorNormalization(data):  # data as numpy array
    min_val = tf.reduce_min(data)  # get min val
    max_val = tf.reduce_max(data)  # get max val
    data_normal = (data - min_val) / (
        max_val - min_val
    )  # get normalized data as numpy array
    minVal = min_val.numpy()  # convert minimum tensor to numpy
    maxVal = max_val.numpy()  # convert maximum tensor to numpy
    return (
        tf.cast(data_normal, tf.float32),
        minVal,
        maxVal,
    )  # tensorarray, float 32 datatype, min, max


## data loading
# All files should be in the same directory (folder)
normal_data_file = "20260421_224126_Brewing_lab8_data.csv"  # normal condition filename: You must change this!
abnormal_data_file = "20260422_202458_Finishing_lab8_data.csv"  # abnormal condition filename: You must chage this!

df_normal = pd.read_csv(normal_data_file)  # normal dataframe
df_abnormal = pd.read_csv(abnormal_data_file)  # abnormal dataframe

frames = [
    df_normal,
    df_abnormal,
]  # frame list to merge two dataframes into one

df = pd.concat(frames)  # new concatenated dataframe

## Data Transformation
# X-axis: 'Xacc array [m/s2]'
# Y-axis: 'Yacc array [m/s2]'
# Z-axis: 'Zacc array [m/s2]'
AXIS = "Yacc array [m/s2]"  # Select one (for your model) of above axes

# Exploding the values contained in selected column and converting the string values into float values
df_new = pd.concat(
    [df["Condition"], df[AXIS].str.split(" ", expand=True).astype(float)],
    axis=1,
)  # transform space delimited array to each value
ds = df_new.copy()  # make ds by copying df

# Converting the Classifier into binary values
ds.loc[df["Condition"] == "Normal", "Status"] = (
    1  # if Condition column is 'Normal', Give 'Status' Column 1
)
ds.loc[df["Condition"] == "Abnormal", "Status"] = (
    0  # if Condition column is 'Abnormal', Give 'Status' Column 0
)
ds.drop(
    "Condition", axis=1, inplace=True
)  # drop 'Condition' column (the first column)

data = ds.values
# Define Raw data W/O signal processing
raw_data = data[:, :-1]
# Labels: The last column
labels = data[:, -1]

time_data = timeFeatures(raw_data)  # define time domain feature
freq_data = freqFeatures(raw_data)  # define frequency domain feature (DFT)

## Data (feature) selection and Split training and validation dataset
# Feature selection
input_feature = freq_data  # raw_data, time_data, or freq_data

## finally print out the min value and the max values of the input feature
print(
    "The minimum is {} and the maximum is {}.".format(
        tensorNormalization(input_feature)[1],
        tensorNormalization(input_feature)[2],
    )
)

The minimum is 0.0 and the maximum is 0.3741513401613265.
